# 10. CODE DEFECTS

- Types of code defects
- Datasets
- Encoders
- LLMs
- Agents
- Exercise
- References

# 1. Types of Code Defects: From Bugs to Security Flaws

## What do we expect from good software?

Two fundamental things:
1.  **It works as intended:** Functionally correct, efficient, and secure.
2.  **It is easy to maintain:** Clear for developers to understand, modify, and extend.

Violations of these expectations are **defects**. The terminology can be confusing, so let's clarify it:

*   **Defect / Fault:** The umbrella term for any imperfection in the code.
*   **Bug / Error:** A defect that causes the software to produce a wrong result or behave incorrectly against its specification.
*   **Failure:** The observable, incorrect behavior of the system during execution (e.g., a crash, wrong output) caused by a bug.
*   **Weakness / Code Smell / Anti-pattern:** A flaw in design or structure that doesn't immediately cause a failure but makes the code hard to maintain, test, and prore prone to bugs in the future. It's a symptom of deeper problems.
*   **Vulnerability:** A specific, security-critical class of bug. It is a defect that can be intentionally exploited to compromise the system's security (confidentiality, integrity, or availability).

## Why is Automated Vulnerability Detection Important?

- Software is everywhere.
- Approximately 2244 cyber attacks per day (more than 800K attacks per year), approximately one attack [every 39 seconds](https://svitla.com/blog/cybersecurity-threats).
- Software vulnerabilities are the main reason for the prevalence of cyber attacks.
- Modern tools [miss](https://doi.org/10.1145/3533767.3534380) from 47\% to 80\% of real vulnerabilities.


> By the way, Kontur has a [Vulnerability Search Program](https://kontur.ru/bugbounty)

## CWE vs CVE

CWE (Common Weakness Enumeration) is a list of types of software weaknesses.

[Top 25 Most Dangerous](https://cwe.mitre.org/top25/archive/2024/2024_cwe_top25.html):

1. CWE-79: Improper Neutralization of Input During Web Page Generation ('Cross-site Scripting')
2. CWE-787: Out-of-bounds Write
3. CWE-89: Improper Neutralization of Special Elements used in an SQL Command ('SQL Injection')
4. CWE-352: Cross-Site Request Forgery (CSRF)
5. CWE-22: Improper Limitation of a Pathname to a Restricted Directory ('Path Traversal')
6. CWE-125: Out-of-bounds Read
7. CWE-78: Improper Neutralization of Special Elements used in an OS Command ('OS Command Injection')
8. CWE-416: Use After Free
9. CWE-862: Missing Authorization
10. CWE-434: Unrestricted Upload of File with Dangerous Type
11. CWE-94: Improper Control of Generation of Code ('Code Injection')
12. CWE-20: Improper Input Validation
13. CWE-77: Improper Neutralization of Special Elements used in a Command ('Command Injection')
14. CWE-287: Improper Authentication
15. CWE-269: Improper Privilege Management
16. CWE-502: Deserialization of Untrusted Data
17. CWE-200: Exposure of Sensitive Information to an Unauthorized Actor
18. CWE-863: Incorrect Authorization
19. CWE-918: Server-Side Request Forgery (SSRF)
20. CWE-119: Improper Restriction of Operations within the Bounds of a Memory Buffer
21. CWE-476: NULL Pointer Dereference
22. CWE-798: Use of Hard-coded Credentials
23. CWE-190: Integer Overflow or Wraparound
24. CWE-400: Uncontrolled Resource Consumption
25. CWE-306: Missing Authentication for Critical Function

CVE (Common Vulnerabilities and Exposures) is a publicly accessible catalog of known security vulnerabilities and exposures.

> For example: `CVE-2021-44228` is the identifier for the critical Remote Code Execution vulnerability in Apache Log4j.
>
> - **`CVE-`**: The standard prefix.
> - **`2021`**: The year the CVE ID was assigned or the vulnerability was disclosed.
> - **`44228`**: A unique, sequential number assigned to the vulnerability for that year.

# 2. Datasets

Effective models depend on high-quality training data. The key requirements are:

1.  **High-Quality Labels:** Accurate, consistent annotations of what constitutes a defect or vulnerability are essential. Poor labels lead to unreliable models.
2.  **Realistic Data:** The code should resemble real-world projects in complexity and style to ensure the model performs well outside the lab.
3.  **Sufficient Size & Diversity:** The dataset must be large and varied enough (in terms of vulnerability types and code patterns) for the model to learn generalizable features, not just memorize examples.

Several public datasets are commonly used in research. They vary significantly in their characteristics:
1. [SARD 2018](https://samate.nist.gov/SARD/)
2. [Draper](https://arxiv.org/abs/1807.04320)
3. [Devign](https://arxiv.org/abs/1909.03496)
4. [D2A](https://arxiv.org/abs/2102.07995)
5. [ReVeal](https://arxiv.org/abs/2009.07235)
6. [SecurityEval](https://doi.org/10.1145/3549035.3561184)
7. [DiverseVul](https://arxiv.org/abs/2304.00409)


Important Considerations:
- **Synthetic vs. Real-World** 
- **Language Focus:** Most datasets focus on **C/C++** due to their prevalence in vulnerability research. **Java** (SecurityEval) and other languages are less common.
- **Variable Quality:** The utility of a dataset depends heavily on its labeling methodology (e.g., automated heuristics vs. expert review) and its balance of defect types.

**WARNING**

Some datasets (especially synthetic ones) contain comments or variable names that explicitly indicate the type of vulnerability.
A model trained on such a dataset will be useless in practice.

![](res/10_cwe457_comment.png)

Dataset quality:
- 20%--71% of vulnerability labels in real-world datasets are [incorrect](https://arxiv.org/abs/2301.05456)
- 17%--99% of examples are [duplicated](https://arxiv.org/abs/2301.05456)
- after training on such datasets in real-world scenarios, the quality of models [drops](https://ieeexplore.ieee.org/abstract/document/9448435) by more than 50%

[Source: [Croft et al. 2023]](https://arxiv.org/abs/2301.05456)

![](res/10_attributes.png)

![](res/10_quality.png)

# 3. Encoders

1. Pre-trained encoder
2. Fine-tune on dataset

## LineVul

In [[Fu Tantithamthavorn 2022]](https://michaelfu1998-create.github.io/papers/linevul.pdf) the authors implement **line-level vulnerability localization**.


**Core Idea:** The intuition is that tokens that are most contributed to the predictions are likely to be vulnerable tokens.  Given a function predicted as vulnerable by LineVul, the authors perform a line-level vulnerability localization by leveraging the self-attention mechanism inside the Transformer architecture to locate the vulnerable lines.

1. **Token Scoring:**: For each subword token, we can summarize the self-attention scores from each of the $12$ Transformer encoder blocks. After obtaining the attention subword-token scores, we can then integrate those scores into line scores.
2. **Line Segmentation:** The authors split a whole function into many lists of tokens (each list of tokens represents a line) by the Newline control character (i.e., `\n`).
3. **Line Ranking:** Finally, for each list of token scores, they summarize it into one attention line score and rank line scores in a descending order.


Let's summarize:

1. tokens that are most contributed to the predictions are likely to be vulnerable tokens
2. for each subword token in the function, the authors summarize the self-attention scores from each of the 12 Transformer encoder blocks
3. integrate those scores into line scores

![](res/10_linevul.png)

# 4. LLMs

[Prompts](https://arxiv.org/abs/2308.12697) for vulnerability detection using ChatGPT:
- basic prompting
- role-based basic prompt
- reverse-question prompt
- prompting with auxiliary information (Data Flow, API Calls)
- Chain-of-Thought prompting

Basic prompting:
> `Is the following program buggy? Please answer Yes or No. [CODE]`

Role-based basic prompt:
> `I want you to act as a vulnerability detection system. My first request is "Is the following program buggy?" Please answer Yes or No. [CODE]`

Reverse-question prompt:
> `I want you to act as a vulnerability detection system. My first request is "Is the following program **correct**?" Please answer Yes or No. [CODE]`

Prompting with auxiliary information:
> `I want you to act as a vulnerability detection system. I will provide you with the original program and the data flow information, and you will act upon them. Is the following program buggy? [CODE]. [Data Flow description].`
>
> ![](res/10_aux.png)
>
> `Data Flow description: The data value of the variable $v_i$ at the $p_i$-th token comes from/is computed by the variable $v_j$ at the $p_j$-th token.`

Chain-of-Thought Prompting:
> `1. Please describe the intent of the given code. [CODE]`
> `2. I want you to act as a vulnerability detection system. Is the above program buggy? Please answer Yes or No.`

Key Drawbacks of Using LLMs for Defect Detection:

- **High Resource Demand:** LLMs require substantial computational power, memory, and energy for both training and inference, making them expensive and environmentally costly to deploy.
- **Latency Limitations:** The inference time for large models can be slow, which may hinder their integration into real-time development workflows or interactive tools.
- **Dependency on Closed Models:** Many state-of-the-art LLMs are proprietary ("black-box") systems.

# 5. Collaborative Agents (CodeAgent)

[Source: [CodeAgent: Collaborative Agents for Software Engineering](https://arxiv.org/abs/2402.02172)]

CodeAgent $-$ a multi-agent-based system for code review.
- Website: [code-agent-new.vercel.app](https://code-agent-new.vercel.app/index.html)
- Demo: [code-agent-new.vercel.app/index.html#demo](https://code-agent-new.vercel.app/index.html#demo)

## Agents

1. **User**: primary responsibilities involve crafting commit messages, managing original files, and supplying necessary input information like commit details and code
2. **CEO**: main responsibilities include being a decision-maker in policy and strategy, a leader managing teams, and an effective communicator with management and employees. Additionally, the agent specializes in summarizing complex code reviews.
3. **CPO** is responsible for assisting CEO and coder to summary code review reports.
4. **CTO** is familiar with various programming languages and skilled in overarching technology strategies.
5. **Reviewer**: code for defects, vulnerabilities, and consistency issues, fixing bugs, and suggesting improvements.
6. **Coder** actively reviews and revises code. The agent takes decisions about code changes and ensure code quality by evaluating code for defects and suggesting improvements.

![](res/10_codeagent_scheme.png)

## Pipeline

1. **Basic Info Sync** demonstrates the basic information confirmation by the CEO, CTO, and Coder. They will cooperate to recognize the modality of the input (e.g., document, code) and language (e.g., Python, Java, and Go).
2. **Code Review** shows the actual code review process: with the help of Coder, Reviewer will write an analysis report about consistency analysis, vulnerability analysis, format analysis, and suggestions for code revision.
3. **Code Alignment** illustrates the potential code revision: according to analysis reports, Coder will align or revise the code if it finds incorrect snippets with the help of Reviewer.
4. **Document** represents the summarizing and writing conclusion for all the stakeholders: Coder cooperates with CTO and CEO to summarize the document and codes about the whole code review.

![](res/10_codeagent_pipeline.png)

## Self-Improving CoT with QA Checker

QA-Checker is an instruct-driven agent, designed to fine-tune the question inside a conversation to drive the generated answer related to the question.

The initial question (task instruction) is represented as $q_0$, and the first answer of the conversation between Reviewer and Coder is represented as $a_0$. If QA-Checker identifies that $a_0$ is inappropriate for $q_0$, it generates additional instructions attached to the original question (task instruction) and combines them to ask agents to further generate a different answer.

![](res/10_codeagent_self-improving_cot.png)

Initial prompt:

> ```I’m the QA-Checker, an AI-driven agent specializing in ensuring quality and coherence in conversational dynamics, particularly in code review discussions at CodeAgent. My primary role involves analyzing and aligning conversations to maintain topic relevance, ensuring that all discussions about code commits and reviews stay focused and on track. As a sophisticated component of the AI system, I apply advanced algorithms, including Chain-of-Thought reasoning and optimization techniques, to evaluate and guide conversational flow. I am adept at identifying and correcting topic drifts, ensuring that every conversation adheres to its intended purpose. My capabilities extend to facilitating clear and effective communication between team members, making me an essential asset in streamlining code review processes and enhancing overall team collaboration and decision-making```

## Vulnerabilities

![](res/10_codeagent_vulnerabilities.png)

# Exercise

Investigate the effectiveness of different prompting-based approaches for detecting CWEs.
1. Choose LLM
2. Choose promp-based approaches
3. Select a dataset (e.g. https://samate.nist.gov/SARD/)
4. Conduct the research
    Draw conclusions

# References

- https://kontur.ru/bugbounty
- https://svitla.com/blog/cybersecurity-threats
- https://doi.org/10.1145/3533767.3534380
- https://dl.acm.org/doi/10.1145/3549035.3561184
- https://samate.nist.gov/SARD/
- https://ieeexplore.ieee.org/abstract/document/9448435
- https://michaelfu1998-create.github.io/papers/linevul.pdf
- https://arxiv.org/abs/1807.04320
- https://arxiv.org/abs/1909.03496
- https://arxiv.org/abs/2009.07235
- https://arxiv.org/abs/2102.07995
- https://arxiv.org/abs/2301.05456
- https://arxiv.org/abs/2304.00409
- https://arxiv.org/abs/2308.12697
- https://arxiv.org/abs/2402.02172